# FINN — CNN W4A4 su MNIST per Pynq-Z2, con triplicazione (TMR) di layer

**Come si usa**
1. Modifica solo la cella *Configurazione*: di solito `BOARD_IP`, `NET_NAME` e `TMR_LAYERS`.
2. `tmr_layers.py` deve stare nella stessa cartella del notebook.
3. Esegui le celle in ordine dall'alto (`Run All`).
4. La cella *Build* lancia Vivado: dura da decine di minuti a oltre un'ora.
5. Alla fine trovi lo zip di deploy (il nome viene stampato): copialo sulla board e segui l'ultima cella.

Flusso: training (Brevitas) → **triplicazione dei layer scelti (TMR)** → export QONNX → streamlining → layer HW → folding → ZynqBuild → zip di deploy.

**TMR, fase 1.** I layer in `TMR_LAYERS` vengono replicati 3 volte: repliche indipendenti, stessi pesi addestrati, stesso ingresso.
Il voter non c'è ancora. Con `TMR_MERGE = 'tap'` la replica 0 prosegue nella rete e le uscite delle repliche 1 e 2 diventano
uscite extra dell'acceleratore, pronte per collegare il voter nella fase successiva.
Con `TMR_LAYERS = []` il notebook produce la rete originale, come prima.

## 1. Configurazione

In [ ]:
import os

WBITS, ABITS = 4, 4                 # bit di pesi / attivazioni
EPOCHS, BATCH_SIZE, LR = 5, 128, 1e-3
INPUT_SHAPE = (1, 1, 28, 28)        # forma dell'ingresso (batch, canali, altezza, larghezza)

# ---- Rete e TMR ----------------------------------------------------------------------------
NET_NAME   = 'SmallCNN'             # rete da usare: una chiave del registro NETS (sezione 3)
TMR_LAYERS = []                     # layer da triplicare; [] = rete originale, senza TMR. Esempi:
                                    #   [1]                    2° layer logico (conv/linear + BN + attivazione)
                                    #   [0, 3]                 più layer, ciascuno triplicato a sé
                                    #   ['features.3']         un singolo modulo, per nome
                                    #   [('features.3', 'features.4', 'features.5')]   blocco consecutivo = 1 unità
TMR_MERGE  = 'tap'                  # 'tap': flusso FINN | 'median': voter software, solo simulazione PyTorch
EXPORT_KWARGS = {}                  # se l'export segnala repliche fuse: {'optimize': False}

# ---- Board e build -------------------------------------------------------------------------
BOARD     = 'Pynq-Z2'
FPGA_PART = 'xc7z020clg400-1'
CLK_NS    = 10.0                    # 100 MHz
BOARD_IP  = '192.168.2.99'          # IP della Pynq-Z2 (default PYNQ)

TARGET_CYCLES = 5000                # cicli/immagine: più basso = più veloce ma più risorse
FIFO_DEPTH    = 32                  # FIFO davanti a ogni MVAU

# checkpoint condiviso da tutte le varianti TMR della stessa rete; BUILD_DIR (per variante) è scelta dopo il TMR
RUN_ROOT = f'./build/{NET_NAME}_w{WBITS}a{ABITS}_pynqz2'
os.makedirs(RUN_ROOT, exist_ok=True)

## 2. Import

In [ ]:
import numpy as np, torch, torch.nn as nn, torchvision
from collections import Counter
from shutil import copy, copytree, make_archive

from brevitas.nn import QuantConv2d, QuantLinear, QuantReLU
from brevitas.export import export_qonnx
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.core.datatype import DataType
from qonnx.custom_op.registry import getCustomOp
from qonnx.transformation.general import (GiveReadableTensorNames, GiveUniqueNodeNames,
                                          RemoveStaticGraphInputs, RemoveUnusedTensors)
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.insert_topk import InsertTopK
from qonnx.transformation.merge_onnx_models import MergeONNXModels
from qonnx.transformation.lower_convs_to_matmul import LowerConvsToMatMul

from finn.util.pytorch import ToTensor
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from finn.transformation.streamline import Streamline
from finn.transformation.streamline.reorder import (MoveScalarLinearPastInvariants, MakeMaxPoolNHWC,
                                                          MoveLinearPastFork)
import finn.transformation.streamline.absorb as absorb
from finn.transformation.move_reshape import RemoveCNVtoFCFlatten
import finn.transformation.fpgadataflow.convert_to_hw_layers as to_hw
from finn.transformation.fpgadataflow.create_dataflow_partition import CreateDataflowPartition
from finn.transformation.fpgadataflow.specialize_layers import SpecializeLayers
from finn.transformation.fpgadataflow.set_folding import SetFolding
from finn.transformation.fpgadataflow.make_zynq_proj import ZynqBuild
from finn.transformation.fpgadataflow.make_pynq_driver import MakePYNQDriver

from tmr_layers import (apply_tmr, logical_layers, TMRExportWrapper, export_tmr, check_replicas,
                        tag_tap_outputs, restore_tap_outputs, strip_tap_postprocessing)

torch.manual_seed(0)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def tidy_up(m):
    for t in [InferShapes(), FoldConstants(), GiveUniqueNodeNames(),
              GiveReadableTensorNames(), InferDataTypes(), RemoveStaticGraphInputs()]:
        m = m.transform(t)
    return m

## 3. Dataset e rete
Input solo ÷255 (niente normalizzazione): così l'FPGA riceve direttamente i pixel UINT8.
Conv/FC senza bias + BatchNorm + ReLU quantizzata: FINN le fonde in un'unica tabella di soglie.

La rete si sceglie con `NET_NAME` dal registro `NETS`. Per usarne un'altra definiscila qui e registrala
(`NETS['MiaRete'] = MiaRete`): costruttore senza argomenti, ingresso di forma `INPUT_SHAPE`.
La cella stampa i **layer logici** della rete: l'indice a sinistra è quello da usare in `TMR_LAYERS`.

In [ ]:
# MNIST: se c'è già in ./data/MNIST/raw lo usa, altrimenti lo scarica lì
MNIST_RAW = './data/MNIST/raw'
MNIST_FILES = ['train-images-idx3-ubyte.gz', 'train-labels-idx1-ubyte.gz',
               't10k-images-idx3-ubyte.gz',  't10k-labels-idx1-ubyte.gz']
have = all(os.path.isfile(f'{MNIST_RAW}/{f}') for f in MNIST_FILES)
print('MNIST già presente' if have else 'MNIST non trovato: lo scarico in ' + MNIST_RAW)

tf = torchvision.transforms.ToTensor()
train_ds = torchvision.datasets.MNIST('./data', train=True,  download=not have, transform=tf)
test_ds  = torchvision.datasets.MNIST('./data', train=False, download=not have, transform=tf)
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_ds,  batch_size=1000)

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            QuantConv2d(1, 16, 3, bias=False, weight_bit_width=WBITS),  nn.BatchNorm2d(16), QuantReLU(bit_width=ABITS),
            QuantConv2d(16, 16, 3, bias=False, weight_bit_width=WBITS), nn.BatchNorm2d(16), QuantReLU(bit_width=ABITS),
            nn.MaxPool2d(2),
            QuantConv2d(16, 32, 3, bias=False, weight_bit_width=WBITS), nn.BatchNorm2d(32), QuantReLU(bit_width=ABITS),
            nn.MaxPool2d(2))
        self.classifier = nn.Sequential(
            QuantLinear(800, 64, bias=False, weight_bit_width=WBITS), nn.BatchNorm1d(64), QuantReLU(bit_width=ABITS),
            QuantLinear(64, 10, bias=False, weight_bit_width=WBITS))

    def forward(self, x):
        return self.classifier(self.features(x).flatten(1))

NETS = {'SmallCNN': SmallCNN}       # registro delle reti disponibili
net = NETS[NET_NAME]().to(DEVICE)

print('Layer logici (indice da usare in TMR_LAYERS):')
for b in logical_layers(net):
    print(' ', b)

## 4. Training
Se esiste già `model.pth` nella cartella della rete (`RUN_ROOT`) lo ricarica (cancellalo per riaddestrare). Atteso ~99%.
Il checkpoint è condiviso da tutte le varianti TMR: si addestra una volta sola e il TMR viene applicato dopo, sulla rete addestrata.

In [ ]:
ckpt = f'{RUN_ROOT}/model.pth'

def evaluate(m=None):
    m = net if m is None else m
    m.eval(); ok = 0
    with torch.no_grad():
        for x, y in test_loader:
            ok += (m(x.to(DEVICE)).argmax(1).cpu() == y).sum().item()
    return ok / len(test_ds)

if os.path.isfile(ckpt):
    net.load_state_dict(torch.load(ckpt, map_location=DEVICE))
else:
    opt = torch.optim.Adam(net.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    for ep in range(EPOCHS):
        net.train()
        for x, y in train_loader:
            opt.zero_grad()
            nn.functional.cross_entropy(net(x.to(DEVICE)), y.to(DEVICE)).backward()
            opt.step()
        sched.step()
        print(f'epoca {ep+1}/{EPOCHS}  acc {evaluate()*100:.2f}%')
    torch.save(net.state_dict(), ckpt)

print(f'Accuratezza test: {evaluate()*100:.2f}%')

## 4b. Triplicazione (TMR)
`apply_tmr` lavora su una **copia** della rete addestrata: ogni layer in `TMR_LAYERS` diventa tre repliche indipendenti
(parametri non condivisi) alimentate dallo stesso ingresso. Con repliche identiche l'accuratezza non deve cambiare: la cella lo verifica.
Da qui in poi i file di build vanno in una cartella dedicata alla variante (`baseline` oppure `tmr-<merge>__<layer>`).

In [ ]:
tmr_net, tmr_report = apply_tmr(net.cpu().eval(), TMR_LAYERS, merge=TMR_MERGE)
print(tmr_report.summary())

if tmr_report.enabled:
    acc_base, acc_tmr = evaluate(net.to(DEVICE)), evaluate(tmr_net.to(DEVICE))
    print(f'Accuratezza: originale {acc_base*100:.2f}%  |  con TMR {acc_tmr*100:.2f}%')
    assert abs(acc_base - acc_tmr) < 1e-4, "con repliche identiche l'accuratezza non deve cambiare"

net, net_hw = net.cpu().eval(), tmr_net.cpu().eval()

BUILD_DIR = f'{RUN_ROOT}/{tmr_report.tag}'      # una cartella per variante
os.makedirs(BUILD_DIR, exist_ok=True)
print('BUILD_DIR =', BUILD_DIR)

## 5. Export e preparazione FINN
Export QONNX → FINN-ONNX, aggiunta di pre-processing (÷255, input UINT8) e post-processing (TopK → indice classe),
poi streamlining (tutto diventa aritmetica intera + soglie).
Con TMR `'tap'` il grafo ha uscite extra (`<layer>_r1`, `<layer>_r2`): sono le uscite delle repliche 1 e 2, la prima uscita resta la classe.

In [ ]:
assert not (tmr_report.enabled and TMR_MERGE == 'median'), (
    "TMR_MERGE='median' serve solo per simulare in PyTorch: il voter min/max non ha un layer HW in FINN. "
    "Per il flusso FINN usa 'tap'.")

path = f'{BUILD_DIR}/export.onnx'
export_net = TMRExportWrapper(net_hw)     # aggiunge le uscite delle repliche 1 e 2 (senza TMR 'tap' non cambia nulla)
export_tmr(export_net, torch.randn(*INPUT_SHAPE), path, export_qonnx, **EXPORT_KWARGS)
check_replicas(ModelWrapper(path), len(export_net.tap_names))   # le repliche non devono essere state fuse dall'export
qonnx_cleanup(path, out_file=path)

model = tidy_up(ModelWrapper(path).transform(ConvertQONNXtoFINN()))
tag_tap_outputs(model, export_net.tap_names)         # MergeONNXModels (sotto) scarterebbe le uscite extra

# pre-processing: /255 dentro la rete, input UINT8
pre = f'{BUILD_DIR}/preproc.onnx'
export_qonnx(ToTensor(), torch.randn(*INPUT_SHAPE), pre)
qonnx_cleanup(pre, out_file=pre)
model = model.transform(MergeONNXModels(ModelWrapper(pre).transform(ConvertQONNXtoFINN())))
model = restore_tap_outputs(model)                   # ricrea le uscite extra delle repliche 1 e 2
model.set_tensor_datatype(model.graph.input[0].name, DataType['UINT8'])

# post-processing: TopK
model = tidy_up(model.transform(InsertTopK(k=1)))

# streamlining
model = model.transform(MoveScalarLinearPastInvariants())
model = model.transform(Streamline())
# fan-out verso le repliche TMR: Mul/Add passano dopo il fork e vengono assorbiti in ogni ramo (senza fork non cambia nulla)
model = model.transform(MoveLinearPastFork())
model = model.transform(Streamline())
model = model.transform(LowerConvsToMatMul())
model = model.transform(MakeMaxPoolNHWC())
model = model.transform(absorb.AbsorbTransposeIntoMultiThreshold())
model = model.transform(MakeMaxPoolNHWC())
model = model.transform(absorb.AbsorbConsecutiveTransposes())
model = model.transform(MoveScalarLinearPastInvariants())
model = model.transform(Streamline())
model = model.transform(absorb.AbsorbConsecutiveTransposes())
model = model.transform(absorb.AbsorbScalarMulAddIntoTopK())
model = strip_tap_postprocessing(model)             # uscite extra = flusso intero della replica (Transpose/Mul sono da host)
model = model.transform(InferDataLayouts())
model = model.transform(RemoveUnusedTensors())
model.save(f'{BUILD_DIR}/streamlined.onnx')
print([n.op_type for n in model.graph.node])
print('uscite del grafo:', [o.name for o in model.graph.output])

## 6. Layer HW, partizione e folding
Ogni gruppo di nodi diventa un layer FINN (MVAU, ConvolutionInputGenerator, StreamingMaxPool, LabelSelect).
`SetFolding` sceglie PE/SIMD per stare sotto `TARGET_CYCLES`.

In [ ]:
model = model.transform(to_hw.InferQuantizedMatrixVectorActivation())
model = model.transform(to_hw.InferLabelSelectLayer())
model = model.transform(to_hw.InferThresholdingLayer())
model = model.transform(to_hw.InferConvInpGen())
model = model.transform(to_hw.InferStreamingMaxPool())
model = model.transform(to_hw.InferDuplicateStreamsLayer())   # fan-out dell'ingresso verso le 3 repliche
model = model.transform(RemoveCNVtoFCFlatten())
model = model.transform(absorb.AbsorbConsecutiveTransposes())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(InferDataLayouts())

# separa la parte che va in FPGA
parent = model.transform(CreateDataflowPartition())
sdp = parent.get_nodes_by_op_type('StreamingDataflowPartition')[0]
model = ModelWrapper(getCustomOp(sdp).get_nodeattr('model'))

# MVAU in HLS (percorso più sicuro), poi scelta implementazioni per la Pynq-Z2
for n in model.graph.node:
    if n.op_type in ('MVAU', 'Thresholding'):
        getCustomOp(n).set_nodeattr('preferred_impl_style', 'hls')
model = model.transform(SpecializeLayers(FPGA_PART))
model = model.transform(GiveUniqueNodeNames())

# folding
model = model.transform(SetFolding(target_cycles_per_frame=TARGET_CYCLES, mvau_wwidth_max=80, two_pass_relaxation=True))
for n in model.graph.node:
    if n.op_type.startswith('MVAU'):
        getCustomOp(n).set_nodeattr('inFIFODepths', [FIFO_DEPTH])
model.save(f'{BUILD_DIR}/folded.onnx')

for n in model.graph.node:
    print(f'{n.name:<36} cicli {getCustomOp(n).get_exp_cycles()}')

print('\nLayer HW:', dict(Counter(n.op_type for n in model.graph.node)))

## 7. Build (Vivado)
`ZynqBuild` genera ogni layer in HLS, li unisce in un unico IP, aggiunge DMA e Zynq PS e produce il bitstream.
`MakePYNQDriver` crea il driver Python per la board. ⏱ Cella lunga.

In [ ]:
model = ModelWrapper(f'{BUILD_DIR}/folded.onnx')
model = model.transform(ZynqBuild(platform=BOARD, period_ns=CLK_NS))
model = model.transform(MakePYNQDriver('zynq-iodma'))
model.save(f'{BUILD_DIR}/built.onnx')

print('Bitfile:', model.get_metadata_prop('bitfile'))
print('HWH    :', model.get_metadata_prop('hw_handoff'))

## 8. Pacchetto di deploy
Crea `deploy_cnn_w4a4_pynqz2.zip` con: `resizer.bit`, `resizer.hwh`, driver PYNQ,
`input.npy` (prima immagine di test, UINT8) ed `expected_label.txt` (la sua etichetta).

In [ ]:
deploy_dir = f'{BUILD_DIR}/deploy'
os.makedirs(deploy_dir, exist_ok=True)
copy(model.get_metadata_prop('bitfile'), deploy_dir)
copy(model.get_metadata_prop('hw_handoff'), deploy_dir)
copytree(model.get_metadata_prop('pynq_driver_dir'), deploy_dir, dirs_exist_ok=True)

ishape = model.get_tensor_shape(model.graph.input[0].name)   # [1, 28, 28, 1]
np.save(f'{deploy_dir}/input.npy', test_ds.data[0].numpy().reshape(ishape).astype(np.uint8))
open(f'{deploy_dir}/expected_label.txt', 'w').write(f'{int(test_ds.targets[0])}\n')

# MNIST dentro il pacchetto (cartella mnist/): validate.py lo trova senza scaricare nulla
os.makedirs(f'{deploy_dir}/mnist', exist_ok=True)
for f in MNIST_FILES:
    copy(f'{MNIST_RAW}/{f}', f'{deploy_dir}/mnist/')

ZIP_NAME = f'deploy_{NET_NAME}_w{WBITS}a{ABITS}_{tmr_report.tag}_pynqz2'
make_archive(ZIP_NAME, 'zip', deploy_dir)
print(f'Creato {ZIP_NAME}.zip:', sorted(os.listdir(deploy_dir)))
print('Uscite dell\'acceleratore, in ordine:', [o.name for o in model.graph.output])

## 9. Come usarlo sulla Pynq-Z2

Sostituisci `<ZIP>` con il nome stampato dalla cella *Pacchetto di deploy* (es. `deploy_SmallCNN_w4a4_baseline_pynqz2`).

**Dal PC** (sostituisci l'IP se diverso da `BOARD_IP`):
```bash
scp <ZIP>.zip xilinx@192.168.2.99:/home/xilinx/
ssh xilinx@192.168.2.99            # password di default: xilinx
```

**Sulla board**:
```bash
sudo su
source /etc/profile.d/pynq_venv.sh
cd /home/xilinx
unzip <ZIP>.zip -d deploy_cnn
cd deploy_cnn
```

**1) Una sola immagine** — classifica `input.npy` e scrive il risultato in `output.npy`.
Con TMR `'tap'` l'acceleratore ha più uscite: `--outputfile` accetta un nome per ciascuna, nello stesso ordine
stampato dalla cella di deploy (prima la classe, poi le repliche), ad es. `--outputfile output.npy r1.npy r2.npy`:
```bash
python3 driver.py --exec_mode=execute --batchsize=1 --bitfile=resizer.bit --inputfile=input.npy
python3 -c "import numpy as np; print('predetto:', np.load('output.npy').flatten())"
cat expected_label.txt              # deve coincidere col numero predetto
```

**2) Velocità** — misura immagini/secondo (risultati in `nw_metrics.txt`):
```bash
python3 driver.py --exec_mode=throughput_test --batchsize=1000 --bitfile=resizer.bit
cat nw_metrics.txt
```

**3) Accuratezza** su tutto il test set MNIST.

Prima controlla che ci siano la libreria `dataset_loading` e i file MNIST nella cartella `mnist/`
del pacchetto. Se manca qualcosa, il comando li installa/scarica (serve internet sulla board solo in quel caso):
```bash
python3 -c "import dataset_loading" 2>/dev/null && echo "dataset_loading OK" \
  || pip3 install git+https://github.com/fbcotter/dataset_loading.git

mkdir -p mnist
for f in train-images-idx3-ubyte train-labels-idx1-ubyte t10k-images-idx3-ubyte t10k-labels-idx1-ubyte; do
  [ -f mnist/$f.gz ] && echo "$f OK" \
    || wget -q -P mnist https://ossci-datasets.s3.amazonaws.com/mnist/$f.gz
done
```
Poi la validazione, indicando la cartella del dataset:
```bash
python3 validate.py --dataset mnist --batchsize 1000 --dataset_root mnist
```